In [20]:
from utils.paths import DATA_DIR
from preprocessing.web_utils import ElementTableProp
import pandas as pd

In [2]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

# ====================== Valence Electron ================================
class ElementTableProp:
    def __init__(self, url, table_index=0, timeout=30):
        self.url = url
        self.table_index = table_index
        self.timeout = timeout

    def read_table(self):
        r = requests.get(self.url, headers={"User-Agent": "Mozilla/5.0"}, timeout=self.timeout)
        r.raise_for_status()
        tables = pd.read_html(StringIO(r.text))
        if self.table_index >= len(tables):
            raise IndexError(f"Requested table {self.table_index}, but only {len(tables)} tables found.")
        return tables[self.table_index].reset_index(drop=True)

    @staticmethod
    def _find_symbol_col(df):
        for c in df.columns:
            if str(c).strip().lower() == "symbol":
                return c
        raise KeyError(f"No Symbol/symbol column found. Columns: {list(df.columns)}")

    def extract_property(
        self,
        value_col,
        out_value_name=None,
        dropna=True,
        numeric=True,
        extract_digits=False,
    ):
        df = self.read_table()

        sym_col = self._find_symbol_col(df)

        out_value_name = out_value_name or value_col

        out = df[[sym_col, value_col]].copy()
        out = out.rename(columns={sym_col: "symbol", value_col: out_value_name})

        if extract_digits:
            out[out_value_name] = (
                out[out_value_name].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0]
            )

        if numeric:
            out[out_value_name] = pd.to_numeric(out[out_value_name], errors="coerce")

        if dropna:
            out = out.dropna(subset=[out_value_name])

        out["symbol"] = out["symbol"].astype(str).str.strip()
        out = out.drop_duplicates(subset=["symbol"]).reset_index(drop=True)

        return out
    @staticmethod
    def _last_numeric(row):
        nums = pd.to_numeric(row, errors="coerce").dropna()
        return nums.iloc[-1] if len(nums) else np.nan

    def extract_valence_electrons(self, legend_col="Legend", name_row_start=1, row_step=3, valence_row_offset=3):
        df = self.read_table()

        if legend_col not in df.columns:
            raise KeyError(f"Column '{legend_col}' not found. Columns: {list(df.columns)}")

        df_name = df.iloc[name_row_start::row_step].copy()
        df_val = df.iloc[valence_row_offset::row_step].copy()

        df_name["symbol"] = df_name[legend_col].astype(str).str.extract(r"\b\d+\s+([A-Z][a-z]?)\b")
        df_name = df_name.reset_index(drop=True)

        df_val["valence"] = df_val.apply(self._last_numeric, axis=1)
        df_val = df_val.reset_index(drop=True)

        out = pd.DataFrame({"symbol": df_name["symbol"], "valence": df_val["valence"]})
        out = out.dropna(subset=["symbol"]).drop_duplicates(subset=["symbol"]).reset_index(drop=True)
        return out
    
    @staticmethod
    def to_map(df, value_col):
        return dict(zip(df["symbol"], df[value_col]))

Atomic Radius

In [5]:
url_atr = "https://en.wikipedia.org/wiki/Atomic_radii_of_the_elements_(data_page)"
wiki_atr = ElementTableProp(url_atr, table_index=0)

df_atomic_radius = wiki_atr.extract_property(
    value_col="Metallic",
    out_value_name="atomic_radius_metallic",
    numeric=True,
    extract_digits=True,   # pulls number out of strings like "145 pm"
    dropna=True
)

df_atomic_radius.head()
atomic_radius_map = ElementTableProp.to_map(df_atomic_radius, "atomic_radius_metallic") # units pm


Electronegativity difference

In [6]:
url_electron = "https://en.wikipedia.org/wiki/Electronegativities_of_the_elements_(data_page)"
wiki_en = ElementTableProp(url_electron, table_index=1)

df_electronegativity = wiki_en.extract_property(
    value_col="electronegativity",
    out_value_name="electronegativity",
    numeric=True,
    extract_digits=False,
    dropna=True
)

df_electronegativity.head()
en_map = ElementTableProp.to_map(df_electronegativity, "electronegativity")

Valence Electron

In [21]:
url_valence = "https://en.wikipedia.org/wiki/Electron_configurations_of_the_elements_(data_page)"
wiki_val = ElementTableProp(url_valence, table_index=0)  # change index if needed

df_valence = wiki_val.extract_valence_electrons(legend_col="Legend")
df_valence.loc[df_valence['symbol']=='Ni','valence'] = 10
df_valence.loc[df_valence['symbol']=='Cu','valence'] = 11

df_valence.head()
df_valence.to_pickle(DATA_DIR/"descriptors_data/Valence_electrons.pkl")

Matminer (Information)

In [6]:
from mp_api.client import MPRester
import pandas as pd
import numpy as np

In [4]:
with MPRester("2EEDlkvr2m8hB8XsKaA2lBGV9Ta57jFV") as mpr:
    docs = mpr.materials.summary.search(
        formula="Cu50Ni50",
        fields=["material_id", "formula_pretty", "band_gap", "is_stable"]
    )

df_mp = pd.DataFrame([d.model_dump() for d in docs])
df_mp.head()

Retrieving SummaryDoc documents: 100%|██████████| 3/3 [00:00<00:00, 28597.53it/s]


,formula_pretty,material_id,is_stable,band_gap,fields_not_requested
0,CuNi,mp-crtdv,False,0.0,"[builder_meta, nsites, elements, nelements, co..."
1,CuNi,mp-cpjpd,False,0.0,"[builder_meta, nsites, elements, nelements, co..."
2,CuNi,mp-crted,False,0.0,"[builder_meta, nsites, elements, nelements, co..."


In [5]:
from matminer.descriptors.composition_features import get_pymatgen_descriptor
import numpy as np

avg_mass = np.mean(get_pymatgen_descriptor('LiFePO4', 'atomic_mass')) 

ModuleNotFoundError: No module named 'matminer.descriptors'